# Experiment 0: short pilots and computing cost

The 32 short pilots cover four bases × two learning-rate extremes × two adaptation modes × two tasks on the first dataset partition. They run for 250 successful updates.

Use them to investigate failures, GPU memory and timing, not to select a winner on held-out performance. Extrapolation from a short run is provisional; table size and monitoring overhead make update costs uneven.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from IPython.display import display
from src.visualize import style
from src.visualize.figures import FigureSaver
from src.visualize import campaign as cp
from src.data.dataset_names import display_frame
from src.visualize.inputs import analysis_root
style.apply()
sink = FigureSaver('experiment0/02_short_pilots')
report = cp.NotebookReport('Experiment 0: short pilots and computing cost')


## 1. Pilot coverage

Check the planned trial denominator and failures before interpreting costs.

In [ ]:
pilots = [cp.load_campaign(0, track, 'pilot') for track in ('pd','lgd')]
for pilot in pilots:
    cp.show(sink, cp.plot_coverage(pilot))
report.add('1. Pilot coverage', '\n\n'.join(cp.coverage_summary(c) for c in pilots))

## 2. Early behavior

Only predefined learning-rate extremes are compared. Panels fix base and adaptation, with equal dataset weighting and explicit gaps for missing milestones.

In [ ]:
for pilot in pilots:
    cp.show(sink, cp.plot_trajectory_pages(pilot))
report.add('2. Early behavior', '\n\n'.join(c.track.upper()+'\n'+cp.effect_summary(cp.endpoint_effects(c)) for c in pilots))

## 3. Time, memory and numerical diagnostics

Separate seconds per update, allocated GPU hours, peak memory, data waiting and skipped updates. These are different bottlenecks, and require different remedies.

In [ ]:
for pilot in pilots:
    cp.show(sink, cp.plot_optimization(pilot, 'train_loss'))
    cp.show(sink, cp.plot_diagnostics(pilot))
    display(display_frame(cp.diagnostics(pilot)).drop(columns=['final_ckpt_path','source_file','train_dataset_ids','test_dataset_ids'], errors='ignore'))
report.add('3. Resource diagnostics', '\n\n'.join(c.track.upper()+'\n'+cp.diagnostics(c).select_dtypes('number').describe().round(4).to_string() for c in pilots))

## 4. Resource decision

Choose measured per-base/per-mode time requests with a margin, accounting for monitoring and setup. Shorter honest requests can fit backfill opportunities; queue priority is not guaranteed by a short request alone.

In [ ]:
report.add('4. Resource decision', 'Use measured pilot costs and existing row-cap probes. Do not increase row caps or launch the main grid from the null-control timing extrapolation.')

## 5. Non-credit retention

Fixed public non-credit datasets remain outside the adaptation corpus. Curves compare every milestone with the same starting checkpoint, rows and monitoring seed. This small panel measures retention on those tables; it does not establish universal absence of forgetting.

In [ ]:
from src.visualize import diagnostics as dg
for current in pilots:
    cp.show(sink, cp.plot_trajectory_pages(current, split='ood'))
report.add('5. Non-credit retention', '\n\n'.join(c.track.upper()+'\n'+cp.effect_summary(cp.endpoint_effects(c, 'ood')) for c in pilots))

## 6. Parameter movement and sampled resources

Milestone tensor summaries complement total norm-weighted drift. Resource plots use one median per trial and omit unavailable counters; sampled device utilization is not precise kernel time or energy.

In [ ]:
for current in pilots:
    cp.show(sink, dg.plot_parameters(current))
    cp.show(sink, dg.plot_resources(current))
report.add('6. Parameters and resources', '\n\n'.join(dg.summary(c) for c in pilots))

## Summary

The following text repeats the sections in order. It is included verbatim in `All_Results.md`.

In [ ]:
print(report.summary(sink))